In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])

In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import pipeline, AutoConfig
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    pipeline_device = torch.device('mps')
else:
    device = 'cpu'
    pipeline_device = -1

print({'selected_device': device})

In [ ]:
dataset = load_dataset('dair-ai/emotion', split='validation')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

print({'split': 'validation', 'num_rows': len(dataset)})
print(dataset[:3])

In [ ]:
model_name = 'bhadresh-savani/distilbert-base-uncased-emotion'
config = AutoConfig.from_pretrained(model_name)

clf = pipeline(
    task='text-classification',
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True
)

id2label = {}
if hasattr(config, 'id2label') and config.id2label is not None:
    for k, v in config.id2label.items():
        id2label[int(k)] = str(v)

label2id = {}
if hasattr(config, 'label2id') and config.label2id is not None:
    label2id = {str(k): int(v) for k, v in config.label2id.items()}

print({'model_name': model_name, 'id2label': id2label})

In [ ]:
canonical_set = set(class_names)
alias_map = {
    'sadness': 'sadness',
    'sad': 'sadness',
    'joy': 'joy',
    'happy': 'joy',
    'love': 'love',
    'anger': 'anger',
    'angry': 'anger',
    'fear': 'fear',
    'scared': 'fear',
    'surprise': 'surprise',
    'surprised': 'surprise'
}

def normalize_label(raw_label):
    label = str(raw_label).strip()
    upper_label = label.upper()
    if upper_label.startswith('LABEL_'):
        idx = int(label.split('_')[-1])
        label = id2label.get(idx, label)
    label = str(label).strip().lower()
    label = alias_map.get(label, label)
    if label not in canonical_set:
        raise ValueError(f'Unrecognized label: {raw_label} -> {label}')
    return label

label_to_id = {name: i for i, name in enumerate(class_names)}
print({'class_names': class_names, 'label_to_id': label_to_id})

In [ ]:
texts = dataset['text']
true_ids = dataset['label']
batch_size = 32

pred_outputs = clf(texts, batch_size=batch_size, top_k=1)

pred_labels = []
pred_scores = []
for item in pred_outputs:
    if isinstance(item, list):
        item = item[0]
    pred_labels.append(normalize_label(item['label']))
    pred_scores.append(float(item['score']))

pred_ids = [label_to_id[label] for label in pred_labels]

results_df = pd.DataFrame({
    'text': texts,
    'true_label': [class_names[i] for i in true_ids],
    'predicted_label': pred_labels,
    'score': pred_scores
})
results_df['correct'] = results_df['true_label'] == results_df['predicted_label']
results_df['char_len'] = results_df['text'].map(len)
results_df['word_len'] = results_df['text'].map(lambda x: len(str(x).split()))

def word_length_bucket(n_words):
    if n_words <= 4:
        return '01_very_short_1_4'
    if n_words <= 8:
        return '02_short_5_8'
    if n_words <= 12:
        return '03_medium_9_12'
    if n_words <= 20:
        return '04_long_13_20'
    return '05_very_long_21_plus'

results_df['length_bucket'] = results_df['word_len'].map(word_length_bucket)

print(results_df.head(10).to_dict(orient='records'))

In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)
cm = confusion_matrix(true_ids, pred_ids)
cm_df = pd.DataFrame(cm, index=[f'true_{c}' for c in class_names], columns=[f'pred_{c}' for c in class_names])

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'validation',
    'num_examples': len(dataset),
    'device': device,
    'accuracy': round(float(accuracy), 6)
})
print(report)
print(cm_df.to_string())

In [ ]:
bucket_stats = []
for bucket_name, group in results_df.groupby('length_bucket', sort=True):
    y_true = group['true_label'].map(label_to_id).to_list()
    y_pred = group['predicted_label'].map(label_to_id).to_list()
    bucket_stats.append({
        'length_bucket': bucket_name,
        'num_examples': int(len(group)),
        'accuracy': round(float(accuracy_score(y_true, y_pred)), 6),
        'avg_words': round(float(group['word_len'].mean()), 3),
        'median_words': float(group['word_len'].median()),
        'avg_chars': round(float(group['char_len'].mean()), 3),
        'avg_confidence': round(float(group['score'].mean()), 6)
    })

bucket_df = pd.DataFrame(bucket_stats).sort_values('length_bucket').reset_index(drop=True)
print(bucket_df.to_string(index=False))

In [ ]:
bucket_label_perf = []
for bucket_name, group in results_df.groupby('length_bucket', sort=True):
    for label in class_names:
        sub = group[group['true_label'] == label]
        support = int(len(sub))
        if support == 0:
            continue
        acc = float((sub['predicted_label'] == sub['true_label']).mean())
        bucket_label_perf.append({
            'length_bucket': bucket_name,
            'label': label,
            'support': support,
            'accuracy': round(acc, 6)
        })

bucket_label_df = pd.DataFrame(bucket_label_perf)
print(bucket_label_df.sort_values(['length_bucket', 'label']).to_string(index=False))

In [ ]:
errors_df = results_df.loc[~results_df['correct'], ['text', 'true_label', 'predicted_label', 'score', 'word_len', 'length_bucket']].copy()
errors_df = errors_df.sort_values(by='score', ascending=False).head(20).reset_index(drop=True)
errors_df['text'] = errors_df['text'].map(lambda x: x if len(x) <= 160 else x[:157] + '...')

print({'num_errors': int((~results_df['correct']).sum()), 'showing_top': len(errors_df)})
print(errors_df.to_string(index=False))